# Pre-class setup and preflight

Complete this notebook before Lecture 1. It checks the environment without displaying or saving your API key.

**You need:** an approved RCD LLM allocation, an active API key, and campus networking or CUVPN.


In [ ]:
from pathlib import Path
import os
import sys

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "course_helpers.py").is_file():
        COURSE_ROOT = candidate.resolve()
        break
else:
    raise RuntimeError("Open this notebook from the course directory or its notebooks/ folder.")

os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
print("Course root:", COURSE_ROOT)


## Open OnDemand settings

Use a standard Jupyter Notebook session with approximately **1 CPU, 4 GB memory, no GPU**. Inference runs through the RCD LLM Service rather than on your allocated node.


## Where computation and data go

<img src="../assets/figures/course-data-flow.svg" alt="Diagram showing a Jupyter notebook sending an HTTPS request through the Clemson network to the RCD API and local model" width="960">

Your notebook performs ordinary Python work locally and sends explicitly constructed requests to the RCD service. The API key authenticates the request but is **not** part of the prompt. The remote model receives the message content and supported media—not arbitrary files from your working directory.

**Knowledge check:** Starting Jupyter on a compute node does not mean the model runs on that node. The course requests inference from the RCD service, so no GPU is requested for the notebook session.


## Build the API client once before hiding it in a helper module

A helper function should not appear as unexplained magic. We first write the
small function that reads the key and constructs an OpenAI-compatible
client. Once its behavior is clear, the same implementation can live in
`course_helpers.py` and later notebooks can import it instead of repeating it.

**Before running:** confirm that you are on the Clemson network or CUVPN.
The next cell may open a hidden prompt, but it never prints the key.


In [ ]:
import getpass
import os
import sys
from openai import OpenAI

LOCAL_RCD_URL = "https://llm.rcd.clemson.edu/v1"

def create_client(base_url=LOCAL_RCD_URL, timeout=300.0):
    api_key = os.getenv("RCD_LLM_API_KEY")
    if not api_key:
        api_key = getpass.getpass("Enter your RCD LLM API key: ").strip()
    if not api_key:
        raise RuntimeError("An RCD LLM API key is required.")
    return OpenAI(api_key=api_key, base_url=base_url, timeout=timeout)

if sys.version_info < (3, 10):
    raise RuntimeError("This course requires Python 3.10 or newer for the MCP SDK.")
print("Python:", sys.version.split()[0])


`create_client()` has one job: combine a secret credential, a base URL, and a
timeout into an SDK client. Authentication is configuration; it must not be
placed in a prompt or saved in notebook output.

We now import the maintained copy from `course_helpers.py`. In a real
project, this is the usual progression: prototype a transparent function in
the notebook, test it, then move stable reusable code into a module.


In [ ]:
from course_helpers import create_client

client = create_client()
print("Local RCD client:", client.base_url)


## Check the exact models used in the lessons

This course intentionally fixes local model IDs so that students compare the
same contracts. Lecture 1 introduces the separately governed OpenAI gateway
only when it is needed, so gateway permissions do not block setup or video work.

The next helper is also shown before import. It reads the catalog and raises
a clear error if a required model is absent. It does **not** infer tool or
thinking support from incomplete metadata.


In [ ]:
def list_model_ids(client):
    return sorted(model.id for model in client.models.list().data)

def require_models(model_ids, client):
    available = set(list_model_ids(client))
    missing = [model_id for model_id in model_ids if model_id not in available]
    if missing:
        raise RuntimeError("Required models are unavailable: " + ", ".join(missing))
    return list(model_ids)


After inspecting those two short functions, import their reusable module
versions and perform the live preflight. A failure here is useful: it tells
us to fix VPN, allocation, key, or model availability before a lecture starts.


In [ ]:
from course_helpers import list_model_ids, require_models

LOCAL_MODELS = [
    "gemma-4-31b-non-it",
    "qwen3-30b-a3b-instruct-fp8",
    "qwen3.5-9b",
    "qwen3-omni-30b-a3b",
]
require_models(LOCAL_MODELS, client=client)
print("Required local models are available:", LOCAL_MODELS)


## Checkpoint

You are ready when the cell above confirms every fixed model ID.

Expected failure messages:

- Missing key → set the environment variable or use the secure prompt.
- Authentication failure → check allocation/key status.
- Connection failure → connect to campus networking or CUVPN.
- Missing model → check the live Models page and contact the instructor;
  do not silently substitute a different model during a controlled lesson.
